# 01 · Trazabilidad con MLflow: registrar, comparar y promover un modelo

**Módulo 6 · Sesión 14** — De modelo a producto

## Objetivos

En los módulos 4 y 5 comparamos una docena de modelos sobre Wine Quality, y los
resultados viven en celdas impresas de varios notebooks. Si mañana alguien pregunta "¿con
qué hiperparámetros, qué datos y qué versión de scikit-learn salió ese 0.589?", hay que
volver a ejecutar. **MLflow** resuelve eso: cada entrenamiento se registra como una
*corrida* (*run*) con sus parámetros, métricas, artefactos y modelo, en un lugar que se
puede consultar, comparar y del que se puede **promover** un modelo a producción.

1. Registrar un experimento completo: los candidatos de los módulos 4 y 5 sobre Wine
   Quality, con los cuatro niveles de reproducibilidad del módulo 1 (semilla, entorno,
   hash de los datos, artefacto) anotados automáticamente.
2. Registrar también lo que importa para desplegar y nunca aparece en una tabla de
   métricas: **tamaño** del modelo y **latencia** de predicción.
3. Comparar las corridas con `search_runs` y con la interfaz web.
4. Promover modelos al **registro** (*Model Registry*) con alias, y cargarlos por nombre.
5. Reproducir una corrida a partir de lo registrado.

La teoría está en `01-trazabilidad-mlflow.md`.

**Paquetes:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `lightgbm`, `mlflow`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-6-mlops-despliegue/notebooks
    %pip install -q mlflow

In [ ]:
import hashlib
import io
import logging
import os
import platform
import subprocess
import time
import warnings

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

import joblib
import lightgbm
import matplotlib.pyplot as plt
import mlflow
import mlflow.lightgbm
import mlflow.sklearn
import numpy as np
import pandas as pd
import sklearn
from lightgbm import LGBMClassifier
from mlflow.models import infer_signature
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import PrecisionRecallDisplay, average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)   # silencia avisos informativos de MLflow
SEMILLA = 42

## 1. Dónde se guarda todo: el servidor de tracking

MLflow separa dos cosas: el **almacén de metadatos** (parámetros, métricas, tags; una base
de datos) y el **almacén de artefactos** (archivos: figuras, modelos; una carpeta o un
bucket). Para trabajar en local basta una base SQLite en la carpeta actual y la carpeta
`mlruns/` para los artefactos. Ambos están en `.gitignore`: lo que se versiona con Git
es el código; lo que produce el código se registra en MLflow.

> Para ver la interfaz web, en otra terminal y desde esta carpeta:
> `uv run mlflow ui --backend-store-uri sqlite:///mlflow.db` y abrir <http://localhost:5000>.

In [ ]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
experimento = mlflow.set_experiment("wine-quality-buena")
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimento:", experimento.name, "· id", experimento.experiment_id, "· artefactos en", experimento.artifact_location)

## 2. Los datos, y su huella

Mismos datos, partición y pliegues que los notebooks 02/04/06 del módulo 4 y 06 del
módulo 5. Y tres cosas que el módulo 1 llamó "niveles de reproducibilidad" y que aquí se
anotan en cada corrida: el **hash** del archivo de datos (si alguien cambia una fila, el
hash cambia), el **commit** de Git del código, y las **versiones** de las librerías.

In [ ]:
RUTA_DATOS = "../datos/wine-quality.csv"
vinos = pd.read_csv(RUTA_DATOS)
vinos["tipo"] = (vinos["tipo"] == "tinto").astype(int)
vinos = vinos.drop_duplicates().reset_index(drop=True)
vinos["buena"] = (vinos["quality"] >= 7).astype(int)
X = vinos.drop(columns=["quality", "buena"])
y = vinos["buena"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMILLA)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

with open(RUTA_DATOS, "rb") as f:
    hash_datos = hashlib.sha256(f.read()).hexdigest()[:12]
try:
    commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip() or "sin-git"
except FileNotFoundError:
    commit = "sin-git"

contexto = {"datos.archivo": RUTA_DATOS, "datos.sha256": hash_datos, "datos.filas_train": len(X_train),
            "codigo.commit": commit, "entorno.python": platform.python_version(),
            "entorno.sklearn": sklearn.__version__, "entorno.lightgbm": lightgbm.__version__, "semilla": SEMILLA}
print(pd.Series(contexto).to_string())

## 3. Una corrida por candidato

La función `registrar` hace lo que en el módulo 4 hacía `evaluar_cv`, y además lo
**anota**: dentro de `mlflow.start_run()` todo lo que se registra queda asociado a esa
corrida. Se registran:

- **Parámetros** (`log_params`): los hiperparámetros del modelo y el contexto de arriba.
- **Métricas** (`log_metrics`): AP y AUC en CV (media y error estándar), y dos que no
  salen en los notebooks anteriores y deciden despliegues: el **tamaño** del modelo
  serializado y la **latencia** por predicción.
- **Artefactos** (`log_artifact` / `log_figure`): la tabla de AP por pliegue y la curva
  precisión-recall.
- El **modelo** (`mlflow.sklearn.log_model`; `mlflow.lightgbm.log_model` para LightGBM),
  con su *firma* (tipos de entrada y salida), un ejemplo de entrada y las dependencias
  exactas para cargarlo. MLflow guarda los modelos de scikit-learn en formato `skops`, que
  —a diferencia de `pickle`— rechaza cargar tipos no declarados; por eso un modelo de
  otra librería usa el *flavor* de esa librería.

In [ ]:
def medir_tamano_y_latencia(modelo, X_muestra, n_rep=20):
    buffer = io.BytesIO()
    joblib.dump(modelo, buffer, compress=3)
    tamano_mb = len(buffer.getvalue()) / 1e6
    lote = X_muestra.iloc[:1000]
    inicio = time.perf_counter()
    for _ in range(n_rep):
        modelo.predict_proba(lote)
    latencia_ms = (time.perf_counter() - inicio) / n_rep / len(lote) * 1000
    return tamano_mb, latencia_ms


def registrar(nombre, modelo, parametros, etiquetas=None, log_model=mlflow.sklearn.log_model):
    with mlflow.start_run(run_name=nombre) as corrida:
        mlflow.set_tags({"modelo.familia": type(modelo).__name__, "curso.modulo": "6", **(etiquetas or {})})
        mlflow.log_params({**parametros, **contexto})

        resultados = cross_validate(modelo, X_train, y_train, cv=cv, scoring=["average_precision", "roc_auc"], return_train_score=False)
        ap, auc = resultados["test_average_precision"], resultados["test_roc_auc"]
        modelo.fit(X_train, y_train)
        tamano_mb, latencia_ms = medir_tamano_y_latencia(modelo, X_train)
        mlflow.log_metrics({"ap_cv": ap.mean(), "ap_cv_ee": ap.std(ddof=1) / np.sqrt(len(ap)),
                            "auc_cv": auc.mean(), "tamano_mb": tamano_mb, "latencia_ms_por_fila": latencia_ms,
                            "segundos_ajuste_cv": resultados["fit_time"].sum()})

        pliegues = pd.DataFrame({"pliegue": range(1, 6), "ap": ap, "auc": auc})
        pliegues.to_csv("ap_por_pliegue.csv", index=False)
        mlflow.log_artifact("ap_por_pliegue.csv")
        os.remove("ap_por_pliegue.csv")

        p_cv = cross_val_predict(modelo, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
        fig, eje = plt.subplots(figsize=(5, 4))
        PrecisionRecallDisplay.from_predictions(y_train, p_cv, ax=eje, name=nombre)
        eje.set_title(f"{nombre} · AP (CV) = {ap.mean():.3f}")
        mlflow.log_figure(fig, "curva_precision_recall.png")
        plt.close(fig)

        firma = infer_signature(X_train, modelo.predict_proba(X_train)[:, 1])
        log_model(modelo, name="modelo", signature=firma, input_example=X_train.iloc[:3],
                  pip_requirements=[f"scikit-learn=={sklearn.__version__}", f"pandas=={pd.__version__}",
                                                   f"lightgbm=={lightgbm.__version__}"])
        print(f"{nombre:<28} AP {ap.mean():.3f} ± {ap.std(ddof=1) / np.sqrt(5):.3f} · {tamano_mb:5.2f} MB · "
              f"{latencia_ms:.4f} ms/fila · run {corrida.info.run_id[:8]}")
        return corrida.info.run_id


candidatos = [
    ("logistica", Pipeline([("esc", StandardScaler()), ("clf", LogisticRegression(max_iter=2000))]), {"C": 1.0}),
    ("hist_gradient_boosting", HistGradientBoostingClassifier(random_state=SEMILLA), {"learning_rate": 0.1, "max_iter": 100}),
    ("hist_gradient_boosting_lento", HistGradientBoostingClassifier(learning_rate=0.03, max_iter=400, random_state=SEMILLA), {"learning_rate": 0.03, "max_iter": 400}),
    ("lightgbm", LGBMClassifier(random_state=SEMILLA, verbose=-1), {"learning_rate": 0.1, "n_estimators": 100, "num_leaves": 31}),
    ("extra_trees_100_hoja3", ExtraTreesClassifier(n_estimators=100, min_samples_leaf=3, random_state=SEMILLA, n_jobs=-1), {"n_estimators": 100, "min_samples_leaf": 3}),
    ("extra_trees_300", ExtraTreesClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1), {"n_estimators": 300, "min_samples_leaf": 1}),
]
ids = {nombre: registrar(nombre, modelo, params, log_model=mlflow.lightgbm.log_model if nombre == "lightgbm" else mlflow.sklearn.log_model)
       for nombre, modelo, params in candidatos}

## 4. Comparar corridas

Todo lo registrado se consulta como un `DataFrame` con `search_runs` — la misma tabla que
muestra la interfaz web, filtrable y ordenable. Y ahora, con dos columnas que ninguna
comparación de los módulos anteriores tenía:

In [ ]:
corridas = mlflow.search_runs(experiment_names=[experimento.name], order_by=["metrics.ap_cv DESC"])
columnas = ["tags.mlflow.runName", "metrics.ap_cv", "metrics.ap_cv_ee", "metrics.auc_cv", "metrics.tamano_mb",
            "metrics.latencia_ms_por_fila", "metrics.segundos_ajuste_cv", "params.datos.sha256", "params.codigo.commit"]
tabla = corridas[columnas].rename(columns=lambda c: c.split(".", 1)[1])
print(tabla.round(4).to_string(index=False))

fig, eje = plt.subplots(figsize=(8, 4.5))
eje.scatter(tabla["tamano_mb"], tabla["ap_cv"], s=60)
for _, fila in tabla.iterrows():
    eje.annotate(fila["mlflow.runName"], (fila["tamano_mb"], fila["ap_cv"]), textcoords="offset points", xytext=(6, 4), fontsize=8)
eje.set_xscale("log")
eje.set_xlabel("tamaño del modelo serializado (MB, escala log)")
eje.set_ylabel("AP en CV")
eje.set_title("Lo que no se ve en una tabla de métricas: la AP cuesta megabytes")
plt.show()

Extra-Trees con 300 árboles sin podar es el mejor en AP (0.589, como en el módulo 4) y
pesa **12 MB**: cada árbol memoriza los 4256 vinos. Con 100 árboles y hojas de 3 filas
pierde dos centésimas y pesa 2 MB. Los dos gradient boosting y la logística pesan menos
de 0.2 MB y predicen en microsegundos. Para un notebook la diferencia no importa; para
una API que se descarga en cada arranque de un contenedor, o que corre en un dispositivo
pequeño, decide.

## 5. El registro de modelos: nombre, versión y alias

Una corrida es un experimento; un **modelo registrado** es una decisión: "este es el
modelo `wine-buena`, versión 3, y la versión que sirve la API es la que tiene el alias
`produccion`". El registro guarda versiones numeradas de un mismo nombre y permite
apuntar a ellas por alias, de modo que el código de la API pide
`models:/wine-buena@produccion` y no cambia cuando se promueve una versión nueva.

Registramos dos: el mejor en AP (alias `campeon`) y el elegido para la API (alias
`produccion`), con la razón anotada.

In [ ]:
cliente = mlflow.MlflowClient()
NOMBRE_MODELO = "wine-buena"

mejor = corridas.iloc[0]
version_campeon = mlflow.register_model(f"runs:/{mejor['run_id']}/modelo", NOMBRE_MODELO)
cliente.set_registered_model_alias(NOMBRE_MODELO, "campeon", version_campeon.version)
cliente.set_model_version_tag(NOMBRE_MODELO, version_campeon.version, "razon", "mejor AP en CV")

fila_api = corridas[corridas["tags.mlflow.runName"] == "hist_gradient_boosting"].iloc[0]
version_api = mlflow.register_model(f"runs:/{fila_api['run_id']}/modelo", NOMBRE_MODELO)
cliente.set_registered_model_alias(NOMBRE_MODELO, "produccion", version_api.version)
cliente.set_model_version_tag(NOMBRE_MODELO, version_api.version, "razon",
                              "0.15 MB y microsegundos por fila; pierde 0.04 de AP frente al campeon")

for v in cliente.search_model_versions(f"name='{NOMBRE_MODELO}'"):
    detalle = cliente.get_model_version(NOMBRE_MODELO, v.version)      # search_model_versions no trae los alias
    print(f"versión {v.version} · corrida {v.run_id[:8]} · alias {list(detalle.aliases)} · {v.tags.get('razon', '')}")

Y la carga por alias, que es lo que hará el código de la API: no sabe qué algoritmo hay
detrás ni en qué corrida se entrenó; pide el modelo de producción y recibe algo con
`predict_proba`.

In [ ]:
modelo_produccion = mlflow.sklearn.load_model(f"models:/{NOMBRE_MODELO}@produccion")
modelo_campeon = mlflow.sklearn.load_model(f"models:/{NOMBRE_MODELO}@campeon")
for alias, m in [("produccion", modelo_produccion), ("campeon", modelo_campeon)]:
    p = m.predict_proba(X_test)[:, 1]
    print(f"{alias:<11} {type(m).__name__:<32} AP en prueba {average_precision_score(y_test, p):.3f} · "
          f"AUC {roc_auc_score(y_test, p):.3f}")

## 6. Reproducir una corrida a partir de lo registrado

La prueba de que el registro sirve: tomar una corrida, leer de ella el hash de los datos,
los hiperparámetros y la semilla, y **volver a obtener el mismo número**.

In [ ]:
corrida = mlflow.get_run(ids["extra_trees_100_hoja3"])
params = corrida.data.params
print("Parámetros registrados:", {k: params[k] for k in ["n_estimators", "min_samples_leaf", "semilla", "datos.sha256", "codigo.commit", "entorno.sklearn"]})

with open(params["datos.archivo"], "rb") as f:
    hash_actual = hashlib.sha256(f.read()).hexdigest()[:12]
print("¿Los datos son los mismos?", hash_actual == params["datos.sha256"])

reproducido = ExtraTreesClassifier(n_estimators=int(params["n_estimators"]), min_samples_leaf=int(params["min_samples_leaf"]),
                                   random_state=int(params["semilla"]), n_jobs=-1)
ap_repro = cross_validate(reproducido, X_train, y_train, cv=cv, scoring="average_precision")["test_score"].mean()
print(f"AP registrada: {corrida.data.metrics['ap_cv']:.6f} · AP reproducida: {ap_repro:.6f} · iguales: {np.isclose(ap_repro, corrida.data.metrics['ap_cv'])}")

Los artefactos de la corrida —la curva PR, la tabla por pliegue, el modelo con su firma y
sus dependencias— están en la carpeta de artefactos y en la interfaz web:

In [ ]:
print("Artefactos de la corrida:", [a.path for a in cliente.list_artifacts(corrida.info.run_id)])
print("Artefactos del modelo:", [a.path for a in cliente.list_artifacts(corrida.info.run_id, "modelo")])

## 7. Del registro a la API

Hay dos formas de servir un modelo registrado:

- **`mlflow models serve -m models:/wine-buena@produccion`** levanta una API genérica
  (`/invocations`) sin escribir código. Cómoda para probar; poco control sobre la
  validación de entradas y el contrato.
- Una **API propia** (`../api/`, con FastAPI) que carga el modelo y define el contrato:
  qué campos entran, con qué rangos, qué sale. Es la que construye la sesión.

La API del curso carga un archivo `joblib` producido por `../api/entrenar_modelo.py`
—que entrena el mismo `HistGradientBoostingClassifier` de la versión `produccion`— para
no depender de MLflow en tiempo de ejecución (una dependencia menos en el contenedor). El
mismo modelo se puede exportar desde el registro:

In [ ]:
paquete = {"modelo": modelo_produccion, "columnas": list(X_train.columns), "umbral": 0.5,
           "version_registro": f"{NOMBRE_MODELO} v{version_api.version}", "datos.sha256": hash_datos}
joblib.dump(paquete, "modelo-desde-registro.joblib", compress=3)
print(f"Exportado modelo-desde-registro.joblib ({os.path.getsize('modelo-desde-registro.joblib') / 1e6:.2f} MB)")
os.remove("modelo-desde-registro.joblib")

## Resumen

| Pregunta | Respuesta |
|---|---|
| ¿Qué es una corrida? | Parámetros + métricas + artefactos + modelo, con el hash de los datos, el commit y las versiones anotados: los cuatro niveles de reproducibilidad del módulo 1, automáticos |
| ¿Qué registrar que no está en las tablas de M4–M5? | Tamaño y latencia: el mejor modelo en AP (Extra-Trees 300) pesa 12 MB; el de la API, 0.15 MB, por 0.04 de AP |
| ¿Cómo se compara? | `search_runs` (un `DataFrame`) o la interfaz web (`mlflow ui`) |
| ¿Qué es el registro? | Versiones numeradas de un modelo con nombre; los alias (`campeon`, `produccion`) desacoplan la API de la versión concreta |
| ¿Se puede reproducir? | Sí: con los parámetros, la semilla y el hash de los datos registrados, la AP sale idéntica |